# 04c – EfficientNet-B0 Fine-tuning

**Szerző:** Magda Ferenc (U5O0BB)  
**Projekt:** Gitár-akkord felismerő szoftver gépi látással  
**Notebook célja:** EfficientNet-B0 fine-tuning két fázisban.

> ℹ️ Ez a notebook a korábbi `04_model.ipynb` átnevezett, naprakész változata.

---

## Notebook sorozat

| Notebook | Modell | Státusz |
|----------|--------|---------|
| `04a` | Hagyományos ML (SVM, RF, XGBoost) | előző |
| `04b` | MobileNetV3, ShuffleNetV2 | előző |
| **`04c`** | **EfficientNet-B0** | **← jelen** |
| `04d` | ResNet-50, EfficientNet-B3 | következő |
| `04e` | ViT-B/16 (opcionális) | tervezett |

---

**Rögzített döntések:**
- Architektúra: **EfficientNet-B0** (torchvision pretrained ImageNet)
- Loss: CrossEntropyLoss(weight=class_weights)
- Optimizer: AdamW  |  Scheduler: CosineAnnealingLR
- Early stopping: patience=10, monitor val_loss
- Checkpoint: legjobb val_acc alapján

## Tartalomjegyzék
1. Könyvtárak és konfiguráció  
2. DataLoader újratöltése  
3. Modell felépítése  
4. Training utilities  
5. Fázis A – Frozen backbone  
6. Fázis B – Fokozatos felolvasztás  
7. Eredmények kiértékelése  
8. Modell exportálás  
9. Összefoglaló és következő lépések


## 1. Könyvtárak és konfiguráció

In [ ]:
import warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision.models as tv_models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import confusion_matrix, classification_report
from PIL import Image

warnings.filterwarnings("ignore")

NOTEBOOK_DIR   = Path.cwd()
PROJECT_ROOT   = NOTEBOOK_DIR.parent
DATA_ROOT      = PROJECT_ROOT / "data"
MANIFEST_PATH  = DATA_ROOT / "split_manifest.csv"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR     = PROJECT_ROOT / "output" / "04_model"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
BATCH_SIZE  = 16
NUM_WORKERS = 0
RANDOM_SEED = 42

LR_A, EPOCHS_A, PATIENCE_A        = 1e-3, 20, 8
LR_B_HEAD, LR_B_BACKBONE          = 1e-4, 1e-5
EPOCHS_B, PATIENCE_B              = 30, 10

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Eszkoz   : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
for p in [DATA_ROOT, MANIFEST_PATH]:
    print(f"{'OK' if p.exists() else 'HIANYZIK'}  {p}")


## 2. DataLoader újratöltése

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
CLASSES     = sorted(manifest["class"].unique())
CLASS2IDX   = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS   = {i: c for c, i in CLASS2IDX.items()}
NUM_CLASSES = len(CLASSES)
manifest["label"] = manifest["class"].map(CLASS2IDX)

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df   = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df  = manifest[manifest["split"] == "test"].reset_index(drop=True)

class_counts  = train_df["class"].value_counts()
class_weights = torch.tensor(
    [len(train_df) / (NUM_CLASSES * class_counts.get(c, 1)) for c in CLASSES],
    dtype=torch.float
).to(DEVICE)

print("Class weights:")
for c, w in zip(CLASSES, class_weights):
    print(f"  {c:<10s}  {w:.4f}")

train_tf = T.Compose([
    T.Resize(256), T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomRotation(15),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
valtest_tf = T.Compose([
    T.Resize(256), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class GuitarChordDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df, self.transform = df.reset_index(drop=True), transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"])

train_loader = DataLoader(GuitarChordDataset(train_df, train_tf),
    BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(GuitarChordDataset(val_df,   valtest_tf),
    BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(GuitarChordDataset(test_df,  valtest_tf),
    BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_df)} kep  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print(f"Train batches: {len(train_loader)}")


## 3. Modell felépítése (EfficientNet-B0)

Az EfficientNet-B0 az ImageNet-en pretrained súlyokkal töltődik be. Az eredeti `classifier` fejét lecseréljük `Dropout → Linear(8)` fejre.

**Fine-tuning stratégia:**
- **Fázis A:** backbone lefagyasztva → csak a classifier tanul
- **Fázis B:** utolsó 3 blokk + classifier felolvad, differential lr-rel

In [ ]:
def build_model(num_classes, freeze_backbone=True):
    model = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.DEFAULT)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

def unfreeze_last_n_blocks(model, n=3):
    blocks = list(model.features.children())
    for block in blocks[-n:]:
        for p in block.parameters():
            p.requires_grad = True

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
total, trainable = count_params(model)
print(f"Osszes parameter    : {total:,}")
print(f"Tanithato (Fazis A) : {trainable:,}  ({100*trainable/total:.1f}%)")


## 4. Training utilities (EarlyStopping, Trainer)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4, checkpoint_path=None):
        self.patience, self.min_delta = patience, min_delta
        self.ckpt_path = checkpoint_path
        self.best_loss, self.counter, self.best_state = float("inf"), 0, None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if self.ckpt_path:
                torch.save(self.best_state, self.ckpt_path)
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
            print(f"  Legjobb modell visszatoltve (val_loss={self.best_loss:.4f})")


def run_epoch(model, loader, criterion, optimizer=None, phase="train"):
    is_train = (phase == "train")
    model.train() if is_train else model.eval()
    total_loss = correct = total = 0
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total


def train_phase(model, train_loader, val_loader, criterion,
                optimizer, scheduler, stopper, n_epochs, label=""):
    history = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc"]}
    t0 = time.time()
    for epoch in range(1, n_epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer, "train")
        vl, va = run_epoch(model, val_loader,   criterion, None,      "val")
        scheduler.step()
        for k, v in zip(history, [tl, vl, ta, va]):
            history[k].append(v)
        print(f"[{label}] Ep {epoch:>3}/{n_epochs}  "
              f"tr_loss={tl:.4f}  tr_acc={ta:.3f}  "
              f"vl_loss={vl:.4f}  vl_acc={va:.3f}  "
              f"({time.time()-t0:.0f}s)")
        if stopper.step(vl, model):
            print(f"  Early stopping triggered (patience={stopper.patience}).")
            break
    stopper.restore_best(model)
    return history

print("Training utilities betoltve.")


## 5. Fázis A – Frozen backbone tanítás

Csak a classifier fej tanul. Cél: a fejet beállítani az akkord-feladat reprezentációira ~10–20 epochon belül.

In [ ]:
criterion   = nn.CrossEntropyLoss(weight=class_weights)
optimizer_a = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                    lr=LR_A, weight_decay=1e-4)
scheduler_a = CosineAnnealingLR(optimizer_a, T_max=EPOCHS_A, eta_min=1e-5)
stopper_a   = EarlyStopping(patience=PATIENCE_A,
                             checkpoint_path=CHECKPOINT_DIR / "best_phase_a.pth")

print("=== FAZIS A – Frozen backbone ===")
history_a = train_phase(model, train_loader, val_loader,
                        criterion, optimizer_a, scheduler_a, stopper_a,
                        n_epochs=EPOCHS_A, label="A")
print(f"\nFazis A kesz. Legjobb val_loss={stopper_a.best_loss:.4f}")


## 6. Fázis B – Fokozatos felolvasztás

Az utolsó 3 EfficientNet blokk felolvad. Differential learning rate: classifier `lr=1e-4`, backbone `lr=1e-5`.

In [ ]:
unfreeze_last_n_blocks(model, n=3)
total_b, trainable_b = count_params(model)
print(f"Tanithato (Fazis B): {trainable_b:,}  ({100*trainable_b/total_b:.1f}%)")

param_groups = [
    {"params": model.features.parameters(),    "lr": LR_B_BACKBONE},
    {"params": model.classifier.parameters(),  "lr": LR_B_HEAD},
]
optimizer_b = AdamW(param_groups, weight_decay=1e-4)
scheduler_b = CosineAnnealingLR(optimizer_b, T_max=EPOCHS_B, eta_min=1e-6)
stopper_b   = EarlyStopping(patience=PATIENCE_B,
                             checkpoint_path=CHECKPOINT_DIR / "best_phase_b.pth")

print("\n=== FAZIS B – Fokozatos felolvasztas ===")
history_b = train_phase(model, train_loader, val_loader,
                        criterion, optimizer_b, scheduler_b, stopper_b,
                        n_epochs=EPOCHS_B, label="B")
print(f"\nFazis B kesz. Legjobb val_loss={stopper_b.best_loss:.4f}")


## 7. Eredmények kiértékelése

In [ ]:
def plot_history(h_a, h_b, save_path):
    tr_loss = h_a["train_loss"] + h_b["train_loss"]
    vl_loss = h_a["val_loss"]   + h_b["val_loss"]
    tr_acc  = h_a["train_acc"]  + h_b["train_acc"]
    vl_acc  = h_a["val_acc"]    + h_b["val_acc"]
    epochs  = list(range(1, len(tr_loss) + 1))
    split   = len(h_a["train_loss"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for ax, tr, vl, title in [
        (ax1, tr_loss, vl_loss, "Loss"),
        (ax2, tr_acc,  vl_acc,  "Accuracy"),
    ]:
        ax.plot(epochs, tr, label="Train")
        ax.plot(epochs, vl, label="Val")
        ax.axvline(split + 0.5, color="gray", linestyle="--", linewidth=1.2, label="A->B hatar")
        ax.set_title(title); ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle("EfficientNet-B0 – Training History", fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Abra elmentve: {save_path}")

plot_history(history_a, history_b, OUTPUT_DIR / "training_history.png")


In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test accuracy: {test_acc:.4f}  ({test_acc*100:.1f}%)\n")
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=3))


In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, fmt, title in [
    (axes[0], cm,      "d",    "Confusion Matrix (abszolut)"),
    (axes[1], cm_norm, ".2f",  "Confusion Matrix (normalt)"),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES,
                ax=ax, linewidths=0.5, linecolor="lightgray")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Predikalt"); ax.set_ylabel("Valos")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle(f"EfficientNet-B0  |  Test acc = {test_acc*100:.1f}%", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Confusion matrix elmentve.")


In [ ]:
per_class_acc = cm_norm.diagonal()
fig, ax = plt.subplots(figsize=(10, 4))
colors = plt.cm.RdYlGn(per_class_acc)
bars = ax.bar(CLASSES, per_class_acc, color=colors)
ax.axhline(test_acc, color="navy", linestyle="--", linewidth=1.5,
           label=f"Atlag ({test_acc:.2f})")
ax.set_ylim(0, 1.15); ax.set_ylabel("Accuracy")
ax.set_title("Per-class Test Accuracy – EfficientNet-B0")
ax.legend()
for bar, val in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{val:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_class_accuracy.png", dpi=120, bbox_inches="tight")
plt.show()


## 8. Modell exportálás

Három formátumban:
- `best_model.pth` – state_dict metadata-val
- `model_full.pth` – teljes modell objektum
- `model.onnx` – ONNX (deployment, inference engine-ekhez)

In [ ]:
state_dict_path = CHECKPOINT_DIR / "best_model.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "class2idx":        CLASS2IDX,
    "idx2class":        IDX2CLASS,
    "num_classes":      NUM_CLASSES,
    "img_size":         IMG_SIZE,
    "imagenet_mean":    IMAGENET_MEAN,
    "imagenet_std":     IMAGENET_STD,
    "test_accuracy":    test_acc,
}, state_dict_path)
print(f"State dict mentve: {state_dict_path}")

full_path = CHECKPOINT_DIR / "model_full.pth"
torch.save(model, full_path)
print(f"Teljes modell mentve: {full_path}")

onnx_path   = CHECKPOINT_DIR / "model.onnx"
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    model.cpu(), dummy_input, str(onnx_path),
    export_params=True, opset_version=17,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch_size"}, "logits": {0: "batch_size"}},
)
model.to(DEVICE)
print(f"ONNX export mentve: {onnx_path}")

for p in [state_dict_path, full_path, onnx_path]:
    print(f"  {p.name:<25s}  {p.stat().st_size / 1024 / 1024:.1f} MB")


## 9. Összefoglaló és következő lépések

### Mit értünk el?
- EfficientNet-B0 fine-tuning két fázisban (frozen + fokozatos felolvasztás)
- CrossEntropyLoss class-weighted loss
- CosineAnnealingLR scheduler + EarlyStopping checkpoint mentéssel
- Confusion matrix + per-class accuracy vizualizáció
- Modell export: `.pth` + ONNX

### Eredmények összefüggésben
A kapott `test_acc` értéket érdemes összevetni:
- `04a` baseline ML eredményekkel (CSV: `output/04a_baseline_ml/baseline_results.csv`)
- `04b` MobileNet eredményekkel (CSV: `output/04b_mobile_cnn/mobile_cnn_results.csv`)

### Következő lépések – `04d_advanced_cnn.ipynb`

| Lépés | Leírás |
|-------|--------|
| 1 | ResNet-50 fine-tuning (azonos kétfázisú stratégiával) |
| 2 | EfficientNet-B3 fine-tuning (nagyobb kapacitás) |
| 3 | Összehasonlítás az összes eddigi modellel |

### Hiperparaméter finomhangolási javaslatok

| Paraméter | Jelenlegi | Próbálandó |
|-----------|-----------|------------|
| BATCH_SIZE | 16 | 32 |
| LR_A | 1e-3 | 5e-4, 3e-3 |
| Unfreeze blokkok | 3 | 2, 4, 5 |
| Dropout | 0.3 | 0.2, 0.4, 0.5 |
| Augmentáció | alap | TrivialAugment, MixUp |
